In [1]:
# !git clone https://github.com/had-A42/Actions-Speak-Louder-than-Words-research.git
# %cd Actions-Speak-Louder-than-Words-research
# !git checkout miftakhutdinov
!git pull origin miftakhutdinov

fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


In [2]:
!pip3 install -r requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [3]:
import sys
sys.path.append('/kaggle/working/Actions-Speak-Louder-than-Words-research')

In [2]:
import sys
sys.path.append('/kaggle/working/Actions-Speak-Louder-than-Words-research')

In [4]:
from typing import Callable, Dict, List, Tuple, Any, Optional
from collections import defaultdict
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm


import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn
import torch.nn.functional as F

from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix


from src.data.dataprep import (transform_indices, verify_time_split, reindex_data, temporal_train_test_split)
from src.data.loaders import (load_ml20m, load_amzn_books, load_yambda, split_and_reindex)
from src.data.evaluation import (evaluate, evaluate_model)
from src.data.utils import (create_masked_tensor, build_q_from_train_targets)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

/usr/local/lib/python3.12/dist-packages/implicit/gpu/__init__.py:28: UserWarning: Disabling GPU support because of 'libcublas.so.13: cannot open shared object file: No such file or directory'
  warnings.warn(


In [5]:
class TransfromerTrainDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        self.samples = []

        for uid, history in histories.items():
            indicies = np.arange(0, len(history), max_seq_len)
            splits = np.split(history[:-1], indicies)[1:]
            targets = np.split(history, indicies+1)[1:]
            
            self.samples.extend(
                {
                    'uid': uid,
                    'history': list(s),
                    'targets': list(t),
                    'length': len(s)
                }
                for s, t in zip(splits, targets) if len(s) > 0 and len(t) > 0
            )

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [6]:
class TransfromerEvalDataset(Dataset):
    def __init__(
        self,
        histories: Dict[Any, List[int]],
        targets: Dict[Any, List[int]],
        max_seq_len: int = 100,
    ) -> None:
        super().__init__()

        
        targets_uids = targets.keys()
        self.samples = [
            {
                'uid': uid,
                'history': history[-max_seq_len:],
                'length': len(history[-max_seq_len:])
            }
            for uid, history in histories.items() if uid in targets_uids and history
        ]

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

In [7]:
def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:

    uids = torch.stack([torch.tensor(b['uid'], dtype=torch.long) for b in batch])
    lengths = torch.stack([torch.tensor(b['length'], dtype=torch.long) for b in batch])
    histories = torch.concat([torch.tensor(b['history'], dtype=torch.long) for b in batch])

    res = {
        'uid': uids,
        'length': lengths,
        'history': histories
        }

    if batch[0].get('targets', None):
        targets = torch.concat([torch.tensor(b['targets'], dtype=torch.long) for b in batch])
        res['targets'] = targets

    return res

In [8]:
!gdown --id 1KzSOO2huM1e8ZlEOa4QzVV-TtW70fmeV -O ml-20m.zip
!gdown --id 1tyKvtBnkDV8hwkx1s3BgT3LoH-s5DA1j -O amzn-books-20m.csv.gz
!gdown --id 11WLpAlB7eQTgwrCQSEJtOIe-qTe6qxuC -O yambda-10m.parquet

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1KzSOO2huM1e8ZlEOa4QzVV-TtW70fmeV
From (redirected): https://drive.google.com/uc?id=1KzSOO2huM1e8ZlEOa4QzVV-TtW70fmeV&confirm=t&uuid=9f6902ef-e336-4378-b14c-be14ee0ebb0b
To: /kaggle/working/ml-20m.zip
100%|████████████████████████████████████████| 199M/199M [00:03<00:00, 63.2MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1tyKvtBnkDV8hwkx1s3BgT3LoH-s5DA1j
From (redirected): https://drive.google.com/uc?id=1tyKvtBnkDV8hwkx1s3BgT3LoH-s5DA1j&confirm=t&uuid=9d0e630d-3b74-4d0

In [9]:
TRAIN_BATCH_SIZE = 128
EVAL_BATCH_SIZE = 128
DEVICE='cuda'

config = {
    "ml": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "ml-20m.zip", # https://grouplens.org/datasets/movielens/20m/
        "col_mapping": {
            "userid": "user_id",
            "movieid": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp",
        },
    },
    "yambda": {
        "max_seq_len": 200,
        "test_quantile": 0.1,
        "interactions_path": "yambda-10m.parquet", # из ноута Кирилла (мб кто подкинет ссылку)
        "col_mapping": {
            "uid": "user_id",
            "item_id": "item_id",
            "timestamp": "timestamp",
        },
    },
    "amzn-books": {
        "max_seq_len": 50,
        "test_quantile": 0.1,
        "interactions_path": "amzn-books-20m.csv.gz", # https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/benchmark/0core/rating_only/Books.csv.gz
        "col_mapping": {
            "user_id": "user_id",
            "parent_asin": "item_id",
            "rating": "feedback",
            "timestamp": "timestamp"
        }
    }
}

In [10]:
ml_df = load_ml20m(config["ml"]["interactions_path"], config=config["ml"])
yambda_df = load_yambda(config["yambda"]["interactions_path"], config=config["yambda"])
amzn_df = load_amzn_books(config["amzn-books"]["interactions_path"], config=config["amzn-books"])

In [11]:
ml_train, ml_test, ml_data_description = split_and_reindex(ml_df, config=config["ml"])
yambda_train, yambda_test, yambda_data_description = split_and_reindex(yambda_df, config=config["yambda"])
amzn_train, amzn_test, amzn_data_description = split_and_reindex(amzn_df, config=config["amzn-books"])

In [12]:
datasets = {
    # "ml": {
    #     "train": ml_train,
    #     "test": ml_test,
    #     "desc": ml_data_description
    # },
    # "yambda": {
    #     "train": yambda_train,
    #     "test": yambda_test,
    #     "desc": yambda_data_description
    # },
    "amzn-books": {
        "train": amzn_train,
        "test": amzn_test,
        "desc": amzn_data_description
    }
}

In [13]:
def build_histories(df: pd.DataFrame, desc: Dict) -> Dict:
    return df.sort_values([desc['users'], desc['timestamp']], ascending=True).groupby(desc['users'])[desc['items']].apply(list).to_dict()

for name, d in datasets.items():
    train, test, desc = d['train'], d['test'], d['desc']
    max_seq_len = config[name]['max_seq_len']

    histories = build_histories(train, desc)
    targets = build_histories(test, desc)
    
    d['histories'] = histories
    d['targets'] = targets  

    d['train_dataset'] = TransfromerTrainDataset(histories=histories, max_seq_len=max_seq_len)
    d['eval_dataset'] = TransfromerEvalDataset(histories=histories, targets=targets, max_seq_len=max_seq_len)

    d['train_dataloader'] = DataLoader(
        dataset=d['train_dataset'],
        batch_size=TRAIN_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        drop_last=True
    )
    d['eval_dataloader'] = DataLoader(
        dataset=d['eval_dataset'],
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        drop_last=False
    )

In [14]:
# статистики по датасетам

stats = []
for name, d in datasets.items():
    train, test, desc = d["train"], d["test"], d["desc"]
    n_train = len(train)
    n_test = len(test)
    n_total = n_train + n_test
    stats.append({
        "dataset": name,
        "n_train": n_train,
        "n_test": n_test,
        "n_total": n_total,
        "train_share": round(n_train / n_total, 3),
        "test_share": round(n_test  / n_total, 3),
        "n_train_users": train[desc["users"]].nunique(),
        "n_test_users": test[desc["users"]].nunique(),
        "n_items": train[desc["items"]].nunique(),
    })

pd.DataFrame(stats)

,dataset,n_train,n_test,n_total,train_share,test_share,n_train_users,n_test_users,n_items
0,amzn-books,10931769,512200,11443969,0.955,0.045,1160538,215463,900095


TopPop

In [15]:
class TopPopRec:
    def __init__(self, n_items: int):
        self.n_items = n_items
        self.popular_items = None

    def fit(self, train_histories: Dict[int, List[int]]):
        all_items = np.concatenate(list(train_histories.values()))
        counts = np.bincount(all_items, minlength=self.n_items)
        self.popular_items = np.argsort(counts)[::-1].tolist()
        return self

    def predict(self, user_ids: List[int], topk: int = 100) -> Dict[int, List[int]]:
        top = self.popular_items[:topk]
        return dict(zip(user_ids, [top] * len(user_ids)))

RandomRec

In [16]:
class RandomRec:
    def __init__(self, n_items: int):
        self.n_items = n_items

    def fit(self, train_histories: Dict[int, List[int]]):
        return self

    def predict(self, user_ids: List[int], topk: int = 100) -> Dict[int, List[int]]:
        rng = np.random.default_rng()
        return {uid: rng.choice(self.n_items, size=topk, replace=False).tolist() for uid in user_ids}

iALS

In [17]:
class iALSRec:
    def __init__(self, n_items: int, factors: int = 64, iterations: int = 20):
        self.n_items = n_items
        self.model = AlternatingLeastSquares(
            factors=factors,
            iterations=iterations,
            use_gpu=False
        )
        self.user_factors = None
        self.item_factors = None

    def fit(self, train_histories: Dict[int, List[int]]):
        n_users = max(train_histories.keys()) + 1
        rows = np.repeat(list(train_histories.keys()), [len(v) for v in train_histories.values()])
        cols = np.concatenate(list(train_histories.values()))
        data = np.ones(len(rows))

        user_item = csr_matrix((data, (rows, cols)), shape=(n_users, self.n_items))
        item_user = user_item.T.tocsr()
        self.model.fit(item_user)

        self.user_factors = self.model.user_factors
        self.item_factors = self.model.item_factors
        return self

    def predict(self, user_ids: List[int], topk: int = 100, batch_size: int = 1000) -> Dict[int, List[int]]:
        result = {}
        for i in range(0, len(user_ids), batch_size):
            batch_uids = user_ids[i:i + batch_size]
            valid = [uid for uid in batch_uids if uid < len(self.user_factors)]
            invalid = [uid for uid in batch_uids if uid >= len(self.user_factors)]
    
            for uid in invalid:
                result[uid] = []
    
            if not valid:
                continue
    
            user_vecs = self.user_factors[valid]
            scores = user_vecs @ self.item_factors.T
            top_k_idx = np.argpartition(-scores, kth=topk, axis=1)[:, :topk]
            top_k_scores = scores[np.arange(len(valid))[:, None], top_k_idx]
            sorted_idx = np.argsort(-top_k_scores, axis=1)
            top_indices = top_k_idx[np.arange(len(valid))[:, None], sorted_idx]
            result.update(dict(zip(valid, top_indices.tolist())))
    
        return result

SASRec

In [18]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.dropout = dropout

        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.n_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, n_heads, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)


    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:

        x = self.qkv(x)

        Q = self.split_heads(x[:, :, :self.d_model])
        K = self.split_heads(x[:, :, self.d_model:2*self.d_model])
        V = self.split_heads(x[:, :, 2*self.d_model:])

        causal_mask = torch.ones(Q.size(-2), K.size(-2), dtype=torch.bool, device=Q.device).tril(diagonal=0).unsqueeze(0).unsqueeze(0)
        padding_mask = mask.unsqueeze(1).unsqueeze(1)
        attn_mask = causal_mask & padding_mask

        attn_output = F.scaled_dot_product_attention(Q, K, V, attn_mask=attn_mask, dropout_p=self.dropout if self.training else 0.0)
        return self.out(self.combine_heads(attn_output))

In [19]:
class MLP(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.0):
        super().__init__()
        self.linear = nn.Linear(d_model, 4 * d_model)
        self.gelu = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.out = nn.Linear(4 * d_model, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len, d_model = x.size()

        x = self.linear(x)
        x = self.gelu(x)
        x = self.dropout(x)
        x = self.out(x)

        return x

In [20]:
class Block(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.0):
        super().__init__()

        self.dropout = dropout

        self.ln1 = nn.LayerNorm(d_model)
        self.csa = CausalSelfAttention(d_model, n_heads, dropout)
        self.dropout1 = nn.Dropout(dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = MLP(d_model, dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(
        self, x: torch.Tensor, mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:

        B, S, D = x.size()

        if mask is None:
            mask = torch.ones(B, S, dtype=torch.bool, device=x.device)

        x1 = self.ln1(x)
        x1 = self.csa(x1, mask)
        x1 = self.dropout1(x1)

        x = x + x1

        x2 = self.ln2(x)
        x2 = self.mlp(x2)
        x2 = self.dropout2(x2)

        x = x + x2


        return x

In [21]:
class GPT(nn.Module):
    def __init__(
        self,
        max_seq_len: int,
        n_layers: int,
        d_model: int,
        n_heads: int,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.dropout = dropout
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.ln = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:

        x = self.drop(x)
        for block in self.blocks:
            x = block(x, mask)
        x = self.ln(x)

        return x

In [22]:
class UserEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()

        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        self.pos_emb = nn.Embedding(max_seq_len, embedding_dim)
        self.GPT = GPT(max_seq_len, n_layers, embedding_dim, n_heads, dropout)



    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:

        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        item_emb = self.item_embeddings(hist)
        pos_emb = self.pos_emb(torch.arange(hist.size(1), dtype=torch.long, device=hist.device))
        x = item_emb + pos_emb
        x[~mask] = 0.0
        x = self.GPT(x, mask)

        return x[mask]

In [23]:
class TrainSASRecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        
        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.num_negatives = num_negatives
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer("q_counts", q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if "weight" in key:
                nn.init.trunc_normal_(
                    value.data,
                    std=initializer_range,
                    a=-2 * initializer_range,
                    b=2 * initializer_range,
                )
            elif "bias" in key:
                nn.init.zeros_(value.data)


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        return self.compute_loss(encoder_output, inputs)


    def compute_loss(
        self, encoder_output: torch.Tensor, inputs: Dict[str, Any]
    ) -> torch.Tensor:

        target_ids = inputs['targets']
        n = target_ids.shape[0]

        negative_pos = torch.randint(0, n, (n, self.num_negatives), device=target_ids.device)
        negative_ids = target_ids[negative_pos]

        candidate_ids = torch.cat([target_ids[:, None], negative_ids], dim=1)
        candidate_emb = self.encoder.item_embeddings(candidate_ids)

        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1)

        log_total = torch.log(self.q_counts.sum().clamp_min(self.eps))
        log_q = torch.log(self.q_counts[negative_ids].clamp_min(self.eps)) - log_total

        correction = torch.cat([torch.zeros(n, 1, device=logits.device), log_q], dim=1)
        logits = logits - correction

        labels = torch.zeros(n, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [24]:
def train(
    dataloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    num_epochs: int,
    device: str = "cuda",
) -> tuple[dict[str, Any], list[float]]:

    model.train()
    epoch_losses = []

    for epoch in tqdm(range(num_epochs), desc="Epochs"):
        total_loss = 0.0
        num_batches = 0

        for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}", leave=False):
            batch = {k: v.to(device) for k, v in batch.items()}

            optimizer.zero_grad()
            loss = model(batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        epoch_loss = total_loss / num_batches
        epoch_losses.append(epoch_loss)
        tqdm.write(f"Epoch {epoch+1} loss: {epoch_loss:.4f}")

    return model.state_dict(), epoch_losses

In [25]:
class EvalSASRecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 100,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        
        self.encoder = UserEncoder(
            num_items=num_items,
            embedding_dim=embedding_dim,
            max_seq_len=max_seq_len,
            n_layers=n_layers,
            n_heads=n_heads,
            dropout=dropout
        )


    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        lengths = inputs['length']
        last_idx = lengths.cumsum(dim=0) - 1
        user_emb = encoder_output[last_idx]

        item_emb = self.encoder.item_embeddings.weight
        scores = user_emb @ item_emb.T

        return scores

GRU4Rec

In [26]:
class GRUEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        n_layers: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.item_embeddings = nn.Embedding(num_items, embedding_dim)
        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=embedding_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        x = self.dropout(self.item_embeddings(hist))

        lengths_cpu = inputs['length'].cpu()
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths_cpu, batch_first=True, enforce_sorted=False)
        output, _ = self.gru(packed)
        output, _ = nn.utils.rnn.pad_packed_sequence(output, batch_first=True)

        return output[mask]

In [27]:
class TrainGRU4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        n_layers: int = 2,
        dropout: float = 0.1,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        self.encoder = GRUEncoder(num_items, embedding_dim, n_layers, dropout)
        self.num_negatives = num_negatives
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer('q_counts', q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if 'weight' in key:
                nn.init.trunc_normal_(value.data, std=initializer_range, a=-2*initializer_range, b=2*initializer_range)
            elif 'bias' in key:
                nn.init.zeros_(value.data)

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        return self.compute_loss(encoder_output, inputs)

    def compute_loss(self, encoder_output: torch.Tensor, inputs: Dict[str, Any]) -> torch.Tensor:
        target_ids = inputs['targets']
        n = target_ids.shape[0]

        negative_pos = torch.randint(0, n, (n, self.num_negatives), device=target_ids.device)
        negative_ids = target_ids[negative_pos]

        candidate_ids = torch.cat([target_ids[:, None], negative_ids], dim=1)
        candidate_emb = self.encoder.item_embeddings(candidate_ids)

        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1)

        log_total = torch.log(self.q_counts.sum().clamp_min(self.eps))
        log_q = torch.log(self.q_counts[negative_ids].clamp_min(self.eps)) - log_total
        correction = torch.cat([torch.zeros(n, 1, device=logits.device), log_q], dim=1)
        logits = logits - correction

        labels = torch.zeros(n, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [28]:
class EvalGRU4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        n_layers: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        
        self.encoder = GRUEncoder(num_items, embedding_dim, n_layers, dropout)

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        encoder_output = self.encoder(inputs)
        lengths = inputs['length']
        last_idx = lengths.cumsum(dim=0) - 1
        
        user_emb = encoder_output[last_idx]
        item_emb = self.encoder.item_embeddings.weight
        
        return user_emb @ item_emb.T

BERT4Rec

In [29]:
class BERTEncoder(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 200,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.mask_token_id = num_items
        self.item_embeddings = nn.Embedding(num_items + 1, embedding_dim) # +1 для [MASK]
        self.pos_emb = nn.Embedding(max_seq_len + 1, embedding_dim) # +1 для [MASK] позиционного энкодера
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=n_heads,
            dim_feedforward=4 * embedding_dim,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers, enable_nested_tensor=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs: Dict[str, torch.Tensor]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        item_emb = self.item_embeddings(hist)
        pos_emb = self.pos_emb(torch.arange(hist.size(1), dtype=torch.long, device=hist.device))
        x = self.dropout(item_emb + pos_emb)
        x[~mask] = 0.0
        pad_mask = ~mask
        x = self.transformer(x, src_key_padding_mask=pad_mask)
        return x, mask

In [30]:
class TrainBERT4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        num_negatives: int,
        q_counts: torch.Tensor,
        max_seq_len: int = 200,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
        mask_prob: float = 0.2,
        eps: float = 1e-12,
    ) -> None:
        super().__init__()
        self.encoder = BERTEncoder(num_items, embedding_dim, max_seq_len, n_layers, n_heads, dropout)
        self.num_negatives = num_negatives
        self.mask_prob = mask_prob
        self.mask_token_id = num_items
        self.eps = eps

        q_counts = q_counts.detach().float().clamp_min(0)
        self.register_buffer('q_counts', q_counts)
        self.init_weights(0.02)

    @torch.no_grad()
    def init_weights(self, initializer_range: float) -> None:
        for key, value in self.named_parameters():
            if 'weight' in key:
                nn.init.trunc_normal_(value.data, std=initializer_range, a=-2*initializer_range, b=2*initializer_range)
            elif 'bias' in key:
                nn.init.zeros_(value.data)

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
    
        rand = torch.rand(hist.shape, device=hist.device)
        masked_positions = (rand < self.mask_prob) & mask
        no_mask_rows = ~masked_positions.any(dim=1)
        last_valid = mask.sum(dim=1) - 1
        masked_positions[no_mask_rows, last_valid[no_mask_rows]] = True
    
        masked_hist = hist.clone()
        masked_hist[masked_positions] = self.mask_token_id
    
        masked_flat = masked_hist[mask]
        modified_inputs = {'history': masked_flat, 'length': inputs['length']}
    
        encoder_output, out_mask = self.encoder(modified_inputs)
    
        target_ids = hist[masked_positions]
        encoder_output_masked = encoder_output[masked_positions]
    
        return self.compute_loss(encoder_output_masked, target_ids)

    def compute_loss(self, encoder_output: torch.Tensor, target_ids: torch.Tensor) -> torch.Tensor:
        n = target_ids.shape[0]
        negative_pos = torch.randint(0, n, (n, self.num_negatives), device=target_ids.device)
        negative_ids = target_ids[negative_pos]

        candidate_ids = torch.cat([target_ids[:, None], negative_ids], dim=1)
        candidate_emb = self.encoder.item_embeddings(candidate_ids)

        logits = (encoder_output[:, None, :] * candidate_emb).sum(dim=-1)

        log_total = torch.log(self.q_counts.sum().clamp_min(self.eps))
        log_q = torch.log(self.q_counts[negative_ids].clamp_min(self.eps)) - log_total
        correction = torch.cat([torch.zeros(n, 1, device=logits.device), log_q], dim=1)
        logits = logits - correction

        labels = torch.zeros(n, dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

In [31]:
class EvalBERT4RecModel(nn.Module):
    def __init__(
        self,
        num_items: int,
        embedding_dim: int,
        max_seq_len: int = 200,
        n_layers: int = 2,
        n_heads: int = 2,
        dropout: float = 0.1,
    ) -> None:
        super().__init__()
        self.encoder = BERTEncoder(num_items, embedding_dim, max_seq_len, n_layers, n_heads, dropout)
        self.mask_token_id = num_items

    def forward(self, inputs: Dict[str, Any]) -> torch.Tensor:
        hist, mask = create_masked_tensor(inputs['history'], inputs['length'])
        B = hist.size(0)

        
        mask_col = torch.full((B, 1), self.mask_token_id, dtype=torch.long, device=hist.device) # добавляем [MASK] в конец каждой истории
        hist_with_mask = torch.cat([hist, mask_col], dim=1)
        mask_with_mask = torch.cat([mask, torch.ones(B, 1, dtype=torch.bool, device=hist.device)], dim=1)

        new_lengths = inputs['length'] + 1
        flat = hist_with_mask[mask_with_mask]
        modified_inputs = {'history': flat, 'length': new_lengths}

        encoder_output, new_mask = self.encoder(modified_inputs)

        
        last_idx = new_lengths.cumsum(dim=0) - 1 # берём скрытое состояние последней позиции ([MASK])
        flat_output = encoder_output[new_mask]
        user_emb = flat_output[last_idx]

        item_emb = self.encoder.item_embeddings.weight[:self.mask_token_id]  # только реальные айтемы
        
        return user_emb @ item_emb.T

In [32]:
results = []

for name, d in datasets.items():
    desc = d['desc']
    user_ids = list(d['targets'].keys())

    # TopPopRec
    toppop_model = TopPopRec(n_items=desc['n_items']).fit(d['histories'])
    for topk in [10, 50, 200]:
        toppop_recs = toppop_model.predict(user_ids, topk=topk)
        toppop_metrics = evaluate(targets=d['targets'], candidates=toppop_recs, catalog_size=desc['n_items'], topk=topk)
        print(f"[{name}] TopPop @{topk}: {toppop_metrics}")
        results.append({'dataset': name, 'model': 'toppop', 'topk': topk, **toppop_metrics})

    del toppop_model
    # RandomRec
    random_model = RandomRec(n_items=desc['n_items']).fit(d['histories'])
    for topk in [10, 50, 200]:
        random_recs = random_model.predict(user_ids, topk=topk)
        random_metrics = evaluate(targets=d['targets'], candidates=random_recs, catalog_size=desc['n_items'], topk=topk)
        print(f"[{name}] Random @{topk}: {random_metrics}")
        results.append({'dataset': name, 'model': 'random', 'topk': topk, **random_metrics})

    del random_model
    # iALS
    # ials_model = iALSRec(n_items=desc['n_items'], factors=64, iterations=20).fit(d['histories'])
    # for topk in [10, 50, 200]:
    #     ials_recs = ials_model.predict(user_ids, topk=topk)
    #     ials_metrics = evaluate(targets=d['targets'], candidates=ials_recs, catalog_size=desc['n_items'], topk=topk)
    #     print(f"[{name}] iALS @{topk}: {ials_metrics}")
    #     results.append({'dataset': name, 'model': 'ials', 'topk': topk, **ials_metrics})

    # del ials_model

    # q_counts

    train_target_ids = torch.tensor([target for idx in range(len(d['train_dataset'])) for target in d['train_dataset'][idx]['targets']], dtype=torch.long)
    q_counts = build_q_from_train_targets(train_target_ids, catalog_size=desc['n_items'])


    # SASRec

    train_model = TrainSASRecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        num_negatives=512,
        q_counts=q_counts,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
        dropout=0.1,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(train_model.parameters(), lr=1e-3)

    checkpoint, losses = train(
        dataloader=d['train_dataloader'],
        model=train_model,
        optimizer=optimizer,
        num_epochs=10,
        device=DEVICE,
    )

    eval_model = EvalSASRecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
    ).to(DEVICE)

    encoder_state = {
        key.removeprefix("encoder."): value
        for key, value in checkpoint.items()
        if key.startswith("encoder.")
    }
    eval_model.encoder.load_state_dict(encoder_state)

    for topk in [10, 50, 200]:
        metrics = evaluate_model(dataloader=d['eval_dataloader'], model=eval_model, catalog_size=desc['n_items'], topk=topk, device=DEVICE, targets=d['targets'], evaluate_fn=evaluate)
        print(f"[{name}] SASRec @{topk}: {metrics}")
        results.append({'dataset': name, 'model': 'sasrec', 'topk': topk, **metrics})

    del train_model, optimizer
    
    # GRU4Rec
    train_gru = TrainGRU4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        num_negatives=512,
        q_counts=q_counts,
        n_layers=2,
        dropout=0.1,
    ).to(DEVICE)
    optimizer_gru = torch.optim.Adam(train_gru.parameters(), lr=1e-3)
    checkpoint_gru, losses_gru = train(
        dataloader=d['train_dataloader'],
        model=train_gru,
        optimizer=optimizer_gru,
        num_epochs=10,
        device=DEVICE,
    )
    eval_gru = EvalGRU4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        n_layers=2,
    ).to(DEVICE)
    encoder_state_gru = {
        key.removeprefix('encoder.'): value
        for key, value in checkpoint_gru.items()
        if key.startswith('encoder.')
    }
    eval_gru.encoder.load_state_dict(encoder_state_gru)
    
    for topk in [10, 50, 200]:
        metrics_gru = evaluate_model(dataloader=d['eval_dataloader'], model=eval_gru, catalog_size=desc['n_items'], topk=topk, device=DEVICE, targets=d['targets'], evaluate_fn=evaluate)
        print(f"[{name}] GRU4Rec @{topk}: {metrics_gru}")
        results.append({'dataset': name, 'model': 'gru4rec', 'topk': topk, **metrics_gru})

    del train_gru, optimizer_gru

    # BERT4Rec
    train_bert = TrainBERT4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        num_negatives=512,
        q_counts=q_counts,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
        dropout=0.1,
        mask_prob=0.2,
    ).to(DEVICE)

    optimizer_bert = torch.optim.Adam(train_bert.parameters(), lr=1e-3)
    checkpoint_bert, losses_bert = train(
        dataloader=d['train_dataloader'],
        model=train_bert,
        optimizer=optimizer_bert,
        num_epochs=10,
        device=DEVICE,
    )

    eval_bert = EvalBERT4RecModel(
        num_items=desc['n_items'],
        embedding_dim=64,
        max_seq_len=config[name]['max_seq_len'],
        n_layers=2,
        n_heads=2,
    ).to(DEVICE)

    encoder_state_bert = {
        key.removeprefix('encoder.'): value
        for key, value in checkpoint_bert.items()
        if key.startswith('encoder.')
    }
    eval_bert.encoder.load_state_dict(encoder_state_bert)

    for topk in [10, 50, 200]:
        metrics_bert = evaluate_model(dataloader=d['eval_dataloader'], model=eval_bert, catalog_size=desc['n_items'], topk=topk, device=DEVICE, targets=d['targets'], evaluate_fn=evaluate)
        print(f"[{name}] BERT4Rec @{topk}: {metrics_bert}")
        results.append({'dataset': name, 'model': 'bert4rec', 'topk': topk, **metrics_bert})

    del train_bert, optimizer_bert

    del eval_model, eval_gru, eval_bert

    
    # del d['train_dataset'], d['eval_dataset']
    # del d['train_dataloader'], d['eval_dataloader']
    # del d['train'], d['test']
    # del d['histories'], d['targets']
    del q_counts, train_target_ids
    torch.cuda.empty_cache()

pd.DataFrame(results)

[amzn-books] TopPop @10: defaultdict(<class 'float'>, {'hitrate': 0.004659732761541425, 'recall': np.float64(0.0022717596929874673), 'ndcg': np.float64(0.0011328359190526623), 'coverage': 1.1109938395391597e-05})
[amzn-books] TopPop @50: defaultdict(<class 'float'>, {'hitrate': 0.01802629685839332, 'recall': np.float64(0.008848225408069399), 'ndcg': np.float64(0.002779691996400565), 'coverage': 5.5549691976957986e-05})
[amzn-books] TopPop @200: defaultdict(<class 'float'>, {'hitrate': 0.03738925012647183, 'recall': np.float64(0.018408811323277607), 'ndcg': np.float64(0.004497800347985698), 'coverage': 0.00022219876790783194})
[amzn-books] Random @10: defaultdict(<class 'float'>, {'hitrate': 2.7847008535108116e-05, 'recall': np.float64(8.740866567964494e-06), 'ndcg': np.float64(4.270102420993508e-06), 'coverage': 0.9084729945172454})
[amzn-books] Random @50: defaultdict(<class 'float'>, {'hitrate': 0.00010674686605124778, 'recall': np.float64(3.613956662689533e-05), 'ndcg': np.float64(1

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

Epoch 1:   0%|          | 0/9075 [00:00<?, ?it/s]

Epoch 1 loss: 18.2303


Epoch 2:   0%|          | 0/9075 [00:00<?, ?it/s]

...

Epoch 9:   0%|          | 0/9075 [00:00<?, ?it/s]

Epoch 9 loss: 18.3744


Epoch 10:   0%|          | 0/9075 [00:00<?, ?it/s]

Epoch 10 loss: 18.0846
[amzn-books] BERT4Rec @10: defaultdict(<class 'float'>, {'hitrate': 0.01648078788469482, 'recall': np.float64(0.007341928260949095), 'ndcg': np.float64(0.004333345717073405), 'coverage': 0.044105344435865106})
[amzn-books] BERT4Rec @50: defaultdict(<class 'float'>, {'hitrate': 0.04777618431006716, 'recall': np.float64(0.022748444250263674), 'ndcg': np.float64(0.008408728645919041), 'coverage': 0.10173037290508224})
[amzn-books] BERT4Rec @200: defaultdict(<class 'float'>, {'hitrate': 0.10057411249263215, 'recall': np.float64(0.053682257870336376), 'ndcg': np.float64(0.014197242895258418), 'coverage': 0.1907709741749482})


,dataset,model,topk,hitrate,recall,ndcg,coverage
0,amzn-books,toppop,10,0.004660,0.002272,0.001133,0.000011
1,amzn-books,toppop,50,0.018026,0.008848,0.002780,0.000056
2,amzn-books,toppop,200,0.037389,0.018409,0.004498,0.000222
3,amzn-books,random,10,0.000028,0.000009,0.000004,0.908473
4,amzn-books,random,50,0.000107,0.000036,0.000013,0.999992
5,amzn-books,random,200,0.000534,0.000265,0.000057,1.000000
6,amzn-books,sasrec,10,0.024018,0.011165,0.007356,0.230702
7,amzn-books,sasrec,50,0.061059,0.030276,0.012373,0.459547
8,amzn-books,sasrec,200,0.118763,0.065845,0.019004,0.679680
9,amzn-books,gru4rec,10,0.025072,0.011750,0.007419,0.114010


In [4]:
data = [
    # ML-20M
    {'dataset': 'ml', 'model': 'random',   'hitrate@10': 0.0383, 'recall@10': 0.0041, 'ndcg@10': 0.0041, 'coverage@10': 0.9372, 'hitrate@50': 0.1610, 'recall@50': 0.0055, 'ndcg@50': 0.0048, 'coverage@50': 1.0000, 'hitrate@200': 0.3758, 'recall@200': 0.0119, 'ndcg@200': 0.0074, 'coverage@200': 1.0000},
    {'dataset': 'ml', 'model': 'toppop',   'hitrate@10': 0.2929, 'recall@10': 0.0705, 'ndcg@10': 0.0730, 'coverage@10': 0.0006, 'hitrate@50': 0.5163, 'recall@50': 0.0765, 'ndcg@50': 0.0705, 'coverage@50': 0.0028, 'hitrate@200': 0.7402, 'recall@200': 0.1528, 'ndcg@200': 0.1026, 'coverage@200': 0.0111},
    {'dataset': 'ml', 'model': 'ials',     'hitrate@10': 0.0022, 'recall@10': 0.0002, 'ndcg@10': 0.0003, 'coverage@10': 0.0281, 'hitrate@50': 0.0052, 'recall@50': 0.0001, 'ndcg@50': 0.0002, 'coverage@50': 0.0983, 'hitrate@200': 0.0125, 'recall@200': 0.0002, 'ndcg@200': 0.0002, 'coverage@200': 0.2935},
    {'dataset': 'ml', 'model': 'bert4rec', 'hitrate@10': 0.2987, 'recall@10': 0.0560, 'ndcg@10': 0.0549, 'coverage@10': 0.1355, 'hitrate@50': 0.6091, 'recall@50': 0.0783, 'ndcg@50': 0.0651, 'coverage@50': 0.2467, 'hitrate@200': 0.8165, 'recall@200': 0.1780, 'ndcg@200': 0.1082, 'coverage@200': 0.4069},
    {'dataset': 'ml', 'model': 'gru4rec',  'hitrate@10': 0.4845, 'recall@10': 0.1063, 'ndcg@10': 0.1053, 'coverage@10': 0.1259, 'hitrate@50': 0.7689, 'recall@50': 0.1305, 'ndcg@50': 0.1123, 'coverage@50': 0.2283, 'hitrate@200': 0.8927, 'recall@200': 0.2560, 'ndcg@200': 0.1626, 'coverage@200': 0.3765},
    {'dataset': 'ml', 'model': 'sasrec',   'hitrate@10': 0.4917, 'recall@10': 0.1100, 'ndcg@10': 0.1082, 'coverage@10': 0.1451, 'hitrate@50': 0.7731, 'recall@50': 0.1348, 'ndcg@50': 0.1148, 'coverage@50': 0.2660, 'hitrate@200': 0.8975, 'recall@200': 0.2629, 'ndcg@200': 0.1667, 'coverage@200': 0.4362},

    # Yambda
    {'dataset': 'yambda', 'model': 'random',   'hitrate@10': 0.0009, 'recall@10': 0.0001, 'ndcg@10': 0.0001, 'coverage@10': 0.9624, 'hitrate@50': 0.0038, 'recall@50': 0.0003, 'ndcg@50': 0.0001, 'coverage@50': 1.0000, 'hitrate@200': 0.0154, 'recall@200': 0.0013, 'ndcg@200': 0.0004, 'coverage@200': 1.0000},
    {'dataset': 'yambda', 'model': 'toppop',   'hitrate@10': 0.0436, 'recall@10': 0.0097, 'ndcg@10': 0.0072, 'coverage@10': 0.0001, 'hitrate@50': 0.1415, 'recall@50': 0.0256, 'ndcg@50': 0.0126, 'coverage@50': 0.0003, 'hitrate@200': 0.2787, 'recall@200': 0.0586, 'ndcg@200': 0.0215, 'coverage@200': 0.0012},
    {'dataset': 'yambda', 'model': 'ials',     'hitrate@10': 0.0008, 'recall@10': 0.0001, 'ndcg@10': 0.0001, 'coverage@10': 0.0126, 'hitrate@50': 0.0037, 'recall@50': 0.0003, 'ndcg@50': 0.0002, 'coverage@50': 0.0341, 'hitrate@200': 0.0150, 'recall@200': 0.0012, 'ndcg@200': 0.0004, 'coverage@200': 0.0810},
    {'dataset': 'yambda', 'model': 'bert4rec', 'hitrate@10': 0.1005, 'recall@10': 0.0200, 'ndcg@10': 0.0165, 'coverage@10': 0.0578, 'hitrate@50': 0.2770, 'recall@50': 0.0533, 'ndcg@50': 0.0276, 'coverage@50': 0.1337, 'hitrate@200': 0.4928, 'recall@200': 0.1318, 'ndcg@200': 0.0494, 'coverage@200': 0.2614},
    {'dataset': 'yambda', 'model': 'gru4rec',  'hitrate@10': 0.1282, 'recall@10': 0.0255, 'ndcg@10': 0.0217, 'coverage@10': 0.0550, 'hitrate@50': 0.3195, 'recall@50': 0.0623, 'ndcg@50': 0.0339, 'coverage@50': 0.1195, 'hitrate@200': 0.5287, 'recall@200': 0.1485, 'ndcg@200': 0.0581, 'coverage@200': 0.2264},
    {'dataset': 'yambda', 'model': 'sasrec',   'hitrate@10': 0.1526, 'recall@10': 0.0316, 'ndcg@10': 0.0262, 'coverage@10': 0.2374, 'hitrate@50': 0.3622, 'recall@50': 0.0746, 'ndcg@50': 0.0407, 'coverage@50': 0.4501, 'hitrate@200': 0.5756, 'recall@200': 0.1746, 'ndcg@200': 0.0688, 'coverage@200': 0.6994},

    # Amazon Books
    {'dataset': 'amzn-books', 'model': 'random',   'hitrate@10': 0.0000, 'recall@10': 0.0000, 'ndcg@10': 0.0000, 'coverage@10': 0.9087, 'hitrate@50': 0.0002, 'recall@50': 0.0001, 'ndcg@50': 0.0000, 'coverage@50': 1.0000, 'hitrate@200': 0.0004, 'recall@200': 0.0002, 'ndcg@200': 0.0000, 'coverage@200': 1.0000},
    {'dataset': 'amzn-books', 'model': 'toppop',   'hitrate@10': 0.0047, 'recall@10': 0.0023, 'ndcg@10': 0.0011, 'coverage@10': 0.0000, 'hitrate@50': 0.0180, 'recall@50': 0.0088, 'ndcg@50': 0.0028, 'coverage@50': 0.0001, 'hitrate@200': 0.0374, 'recall@200': 0.0184, 'ndcg@200': 0.0045, 'coverage@200': 0.0002},
    {'dataset': 'amzn-books', 'model': 'bert4rec', 'hitrate@10': 0.0165, 'recall@10': 0.0073, 'ndcg@10': 0.0043, 'coverage@10': 0.0441, 'hitrate@50': 0.0478, 'recall@50': 0.0227, 'ndcg@50': 0.0084, 'coverage@50': 0.1017, 'hitrate@200': 0.1006, 'recall@200': 0.0537, 'ndcg@200': 0.0142, 'coverage@200': 0.1908},
    {'dataset': 'amzn-books', 'model': 'gru4rec',  'hitrate@10': 0.0251, 'recall@10': 0.0117, 'ndcg@10': 0.0074, 'coverage@10': 0.1140, 'hitrate@50': 0.0666, 'recall@50': 0.0336, 'ndcg@50': 0.0132, 'coverage@50': 0.2519, 'hitrate@200': 0.1301, 'recall@200': 0.0731, 'ndcg@200': 0.0205, 'coverage@200': 0.4391},
    {'dataset': 'amzn-books', 'model': 'sasrec',   'hitrate@10': 0.0240, 'recall@10': 0.0112, 'ndcg@10': 0.0074, 'coverage@10': 0.2307, 'hitrate@50': 0.0611, 'recall@50': 0.0303, 'ndcg@50': 0.0124, 'coverage@50': 0.4595, 'hitrate@200': 0.1188, 'recall@200': 0.0658, 'ndcg@200': 0.0190, 'coverage@200': 0.6797},
]

pd.DataFrame(data)

,dataset,model,hitrate@10,recall@10,ndcg@10,coverage@10,hitrate@50,recall@50,ndcg@50,coverage@50,hitrate@200,recall@200,ndcg@200,coverage@200
0,ml,random,0.0383,0.0041,0.0041,0.9372,0.1610,0.0055,0.0048,1.0000,0.3758,0.0119,0.0074,1.0000
1,ml,toppop,0.2929,0.0705,0.0730,0.0006,0.5163,0.0765,0.0705,0.0028,0.7402,0.1528,0.1026,0.0111
2,ml,ials,0.0022,0.0002,0.0003,0.0281,0.0052,0.0001,0.0002,0.0983,0.0125,0.0002,0.0002,0.2935
3,ml,bert4rec,0.2987,0.0560,0.0549,0.1355,0.6091,0.0783,0.0651,0.2467,0.8165,0.1780,0.1082,0.4069
4,ml,gru4rec,0.4845,0.1063,0.1053,0.1259,0.7689,0.1305,0.1123,0.2283,0.8927,0.2560,0.1626,0.3765
5,ml,sasrec,0.4917,0.1100,0.1082,0.1451,0.7731,0.1348,0.1148,0.2660,0.8975,0.2629,0.1667,0.4362
6,yambda,random,0.0009,0.0001,0.0001,0.9624,0.0038,0.0003,0.0001,1.0000,0.0154,0.0013,0.0004,1.0000
7,yambda,toppop,0.0436,0.0097,0.0072,0.0001,0.1415,0.0256,0.0126,0.0003,0.2787,0.0586,0.0215,0.0012
8,yambda,ials,0.0008,0.0001,0.0001,0.0126,0.0037,0.0003,0.0002,0.0341,0.0150,0.0012,0.0004,0.0810
9,yambda,bert4rec,0.1005,0.0200,0.0165,0.0578,0.2770,0.0533,0.0276,0.1337,0.4928,0.1318,0.0494,0.2614
